# 🔧 LLM Fine-Tuning & Hyperparameter Tuning Notebook

**Product of:** Knatware Technology
**Developed by:** Kayode Okosi — LLM Developer

---

This notebook provides a complete, well-commented pipeline for **fine-tuning and
tuning open-source Large Language Models (LLMs)** on Google Colab using
Hugging Face `transformers`, `datasets`, `peft` (LoRA), and `trl`.

### What this notebook does
1. Installs and configures the environment (GPU check, dependencies).
2. Loads a base model + tokenizer from the Hugging Face Hub.
3. Loads and preprocesses a training dataset.
4. Configures **LoRA** (Low-Rank Adaptation) for efficient fine-tuning.
5. Sets up training / hyperparameter-tuning arguments.
6. Trains the model and logs metrics.
7. Saves and (optionally) merges + pushes the fine-tuned model.
8. Runs a quick inference test on the fine-tuned model.

### ⚠️ Compulsory parameters
Throughout this notebook, cells that require you to supply a value are marked
with a `# COMPULSORY:` comment. Search for `COMPULSORY` in this notebook to
find every value you must set before running.

### Recommended Colab runtime
`Runtime > Change runtime type > T4 GPU` (or better, e.g. A100/L4 if available
on Colab Pro) — fine-tuning LLMs requires a GPU.

---


## 1. Environment Setup

Check the GPU assigned by Colab and install the required libraries.

In [ ]:
# Check which GPU Colab has assigned to this runtime.
# If this errors out or shows "no GPU", go to:
#   Runtime > Change runtime type > Hardware accelerator > GPU
!nvidia-smi


In [ ]:
# ------------------------------------------------------------------------
# Install dependencies
# ------------------------------------------------------------------------
# transformers  -> model + tokenizer loading, Trainer API
# datasets      -> loading and preprocessing training data
# accelerate    -> multi-GPU / mixed-precision training backend
# peft          -> Parameter-Efficient Fine-Tuning (LoRA, QLoRA, etc.)
# trl           -> SFTTrainer (Supervised Fine-Tuning Trainer), simplifies the training loop
# bitsandbytes  -> 8-bit / 4-bit quantization (needed for QLoRA on limited-VRAM GPUs)
# evaluate      -> metrics (e.g. accuracy, perplexity) during evaluation
# wandb         -> optional experiment tracking / hyperparameter tuning dashboard
!pip install -q -U transformers datasets accelerate peft trl bitsandbytes evaluate wandb sentencepiece


In [ ]:
import os
import torch
import numpy as np

from datasets import load_dataset
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    BitsAndBytesConfig,
    TrainingArguments,
)
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training, PeftModel
from trl import SFTTrainer, SFTConfig

# Confirm GPU is visible to PyTorch
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU device:", torch.cuda.get_device_name(0))


## 2. (Optional) Hugging Face & Weights & Biases Login

* Hugging Face login is required if you are using a **gated model** (e.g. Llama,
  Gemma) or want to **push your fine-tuned model** to the Hub.
* W&B login is optional and only needed if you want live dashboards for
  hyperparameter tuning / experiment tracking.


In [ ]:
from huggingface_hub import login

# COMPULSORY (only if using a gated model or pushing to the Hub):
# Get your token from https://huggingface.co/settings/tokens
# Uncomment and run the line below, then paste your token when prompted.
# login()


In [ ]:
import wandb

# OPTIONAL: enables live loss/metric tracking during training and tuning.
# Get a free API key from https://wandb.ai/authorize
# Uncomment to enable:
# wandb.login()

# If you don't want to use W&B at all, set this environment variable
# so the Trainer doesn't try to log to it:
os.environ["WANDB_DISABLED"] = "true"


## 3. Core Configuration

Set all the core (compulsory) parameters for this fine-tuning run in one place.
Everything downstream reads from this config block, so this is the main section
you need to edit for your own project.


In [ ]:
# =============================================================================
# COMPULSORY CONFIGURATION BLOCK
# Edit every value in this cell before running the rest of the notebook.
# =============================================================================

# COMPULSORY: Base model to fine-tune.
# Must be a valid Hugging Face model repo id (e.g. "meta-llama/Llama-3.2-1B",
# "mistralai/Mistral-7B-v0.1", "google/gemma-2-2b", "Qwen/Qwen2.5-1.5B").
# Smaller models (1B-3B) are recommended for free-tier Colab GPUs.
MODEL_NAME = "Qwen/Qwen2.5-1.5B-Instruct"

# COMPULSORY: Training dataset.
# Must be a valid Hugging Face dataset repo id, OR a local/uploaded file path
# (e.g. "my_data.jsonl") loaded via load_dataset("json", data_files=...).
DATASET_NAME = "mlabonne/guanaco-llama2-1k"

# COMPULSORY: Name of the text field in the dataset that SFTTrainer should
# train on. If your dataset has multiple fields (instruction/response, etc.)
# you must format them into a single "text" field first (see Section 4).
DATASET_TEXT_FIELD = "text"

# COMPULSORY: Directory where checkpoints and the final fine-tuned model
# are saved. This will be created automatically if it doesn't exist.
OUTPUT_DIR = "./knatware-finetuned-model"

# COMPULSORY: Maximum sequence length (in tokens) used for training.
# Longer sequences use more GPU memory. 512-1024 is a safe range for
# free-tier Colab GPUs.
MAX_SEQ_LENGTH = 1024

# COMPULSORY: Whether to load the base model in 4-bit precision (QLoRA).
# Strongly recommended = True on free Colab GPUs (T4 has only ~15GB VRAM).
USE_4BIT = True

print("Configuration set:")
print(f"  MODEL_NAME          = {MODEL_NAME}")
print(f"  DATASET_NAME         = {DATASET_NAME}")
print(f"  DATASET_TEXT_FIELD   = {DATASET_TEXT_FIELD}")
print(f"  OUTPUT_DIR           = {OUTPUT_DIR}")
print(f"  MAX_SEQ_LENGTH       = {MAX_SEQ_LENGTH}")
print(f"  USE_4BIT             = {USE_4BIT}")


## 4. Load & Preprocess the Dataset

Loads the dataset defined above and (if needed) formats it into a single
`text` field that `SFTTrainer` can consume directly.


In [ ]:
# Load the dataset from the Hugging Face Hub.
# If instead you have your own local file, replace this line with e.g.:
#   dataset = load_dataset("json", data_files="my_data.jsonl", split="train")
#   dataset = load_dataset("csv", data_files="my_data.csv", split="train")
dataset = load_dataset(DATASET_NAME, split="train")

print(dataset)
print("\nSample record:")
print(dataset[0])


In [ ]:
# ------------------------------------------------------------------------
# OPTIONAL formatting function
# ------------------------------------------------------------------------
# If your dataset does NOT already contain a single "text" column formatted
# as a full prompt/response string, define a formatting function here that
# converts each example into the format your model expects (e.g. chat
# template). SFTTrainer will call this automatically if provided.
#
# Uncomment and adapt this template if your dataset has separate
# "instruction" / "response" (or similar) columns instead of "text":

# def formatting_func(example):
#     return f"""### Instruction:
# {example['instruction']}
#
# ### Response:
# {example['response']}"""

# If your DATASET_TEXT_FIELD already exists and is fully formatted, you can
# leave formatting_func as None.
formatting_func = None


## 5. Load Tokenizer & Base Model

Loads the tokenizer and base model defined in `MODEL_NAME`, optionally in
4-bit precision (QLoRA) to fit on free-tier Colab GPUs.


In [ ]:
# ------------------------------------------------------------------------
# Tokenizer
# ------------------------------------------------------------------------
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, trust_remote_code=True)

# Most causal LMs don't define a pad token by default; reuse EOS as PAD.
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

tokenizer.padding_side = "right"  # recommended for causal LM fine-tuning


In [ ]:
# ------------------------------------------------------------------------
# 4-bit (QLoRA) quantization config
# ------------------------------------------------------------------------
# load_in_4bit             -> loads model weights in 4-bit precision to save VRAM
# bnb_4bit_quant_type       -> "nf4" is the recommended quant type for LLMs
# bnb_4bit_compute_dtype    -> compute dtype used during forward/backward pass
# bnb_4bit_use_double_quant -> further reduces memory via nested quantization
bnb_config = BitsAndBytesConfig(
    load_in_4bit=USE_4BIT,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True,
)

# ------------------------------------------------------------------------
# Load the base model
# ------------------------------------------------------------------------
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config=bnb_config if USE_4BIT else None,
    device_map="auto",          # automatically place model on available GPU(s)
    trust_remote_code=True,
)

model.config.use_cache = False       # required for gradient checkpointing during training
model.config.pretraining_tp = 1      # disables tensor parallel rank splitting (safe default)

if USE_4BIT:
    model = prepare_model_for_kbit_training(model)


## 6. Configure LoRA (Parameter-Efficient Fine-Tuning)

LoRA freezes the base model weights and trains small low-rank "adapter"
matrices instead — this drastically reduces GPU memory and training time
while achieving results close to full fine-tuning.


In [ ]:
# =============================================================================
# COMPULSORY: LoRA hyperparameters
# These are the primary "tuning" knobs for LoRA fine-tuning.
# =============================================================================

# COMPULSORY: LoRA rank. Controls the size of the trainable adapter matrices.
# Higher = more capacity but more memory/compute. Common values: 8, 16, 32, 64.
LORA_R = 16

# COMPULSORY: LoRA scaling factor. Typically set to 2x LORA_R.
LORA_ALPHA = 32

# COMPULSORY: Dropout applied to LoRA layers, helps prevent overfitting.
LORA_DROPOUT = 0.05

# COMPULSORY: Which linear layers to attach LoRA adapters to.
# For most decoder-only LLMs, attention projection layers are targeted.
# Adjust based on your model architecture if needed (check model.named_modules()).
LORA_TARGET_MODULES = ["q_proj", "k_proj", "v_proj", "o_proj"]

lora_config = LoraConfig(
    r=LORA_R,
    lora_alpha=LORA_ALPHA,
    lora_dropout=LORA_DROPOUT,
    target_modules=LORA_TARGET_MODULES,
    bias="none",
    task_type="CAUSAL_LM",
)

model = get_peft_model(model, lora_config)

# Prints how many parameters are actually trainable vs. total —
# with LoRA this is typically <1-2% of total model parameters.
model.print_trainable_parameters()


## 7. Training & Hyperparameter-Tuning Arguments

These are the main hyperparameters you will experiment with when **tuning**
the model. Each compulsory field is commented with its purpose and typical
value range.


In [ ]:
# =============================================================================
# COMPULSORY: Training / hyperparameter-tuning configuration
# =============================================================================
training_args = SFTConfig(

    # COMPULSORY: where checkpoints + logs are written (must match OUTPUT_DIR).
    output_dir=OUTPUT_DIR,

    # COMPULSORY: number of full passes over the training dataset.
    # 1-3 epochs is typical for instruction fine-tuning to avoid overfitting.
    num_train_epochs=3,

    # COMPULSORY: batch size per GPU device. Reduce if you hit CUDA OOM errors.
    per_device_train_batch_size=2,

    # COMPULSORY: number of steps to accumulate gradients before an optimizer
    # step. Effective batch size = per_device_train_batch_size * this value.
    # Use this to simulate a larger batch size on limited VRAM.
    gradient_accumulation_steps=4,

    # COMPULSORY: optimizer. "paged_adamw_32bit" is memory-efficient and
    # recommended when using 4-bit/8-bit quantized models.
    optim="paged_adamw_32bit",

    # COMPULSORY: how often (in steps) to log training metrics.
    logging_steps=10,

    # COMPULSORY: how often (in steps) to save a checkpoint.
    save_steps=50,

    # COMPULSORY: learning rate — the single most impactful tuning parameter.
    # Typical LoRA fine-tuning range: 1e-4 to 3e-4.
    learning_rate=2e-4,

    # COMPULSORY: max gradient norm for gradient clipping (prevents exploding
    # gradients).
    max_grad_norm=0.3,

    # COMPULSORY: fraction of total training steps used to linearly warm up
    # the learning rate from 0 to the target learning_rate.
    warmup_ratio=0.03,

    # COMPULSORY: learning-rate schedule shape after warmup.
    # "cosine" smoothly decays LR to 0; "constant" and "linear" are alternatives.
    lr_scheduler_type="cosine",

    # COMPULSORY: use mixed precision (bf16) if the GPU supports it (A100/T4
    # support bf16; older GPUs may need fp16=True, bf16=False instead).
    bf16=True,
    fp16=False,

    # COMPULSORY: reduces memory usage by not storing all activations during
    # the forward pass (recomputes them during backward pass instead).
    gradient_checkpointing=True,

    # COMPULSORY: maximum tokenized sequence length (must match MAX_SEQ_LENGTH).
    max_length=MAX_SEQ_LENGTH,

    # COMPULSORY: which column in the dataset holds the training text
    # (must match DATASET_TEXT_FIELD).
    dataset_text_field=DATASET_TEXT_FIELD,

    # Disables external experiment-tracking integrations by default.
    # Set to "wandb" if you logged in above and want live dashboards.
    report_to="none",
)

print("Training arguments configured.")


## 8. Build the Trainer and Start Fine-Tuning

In [ ]:
trainer = SFTTrainer(
    model=model,
    args=training_args,
    train_dataset=dataset,
    formatting_func=formatting_func,   # None if dataset already has a "text" field
    processing_class=tokenizer,
    peft_config=lora_config,
)


In [ ]:
# Start training. This is the long-running cell — progress (loss, learning
# rate, etc.) will print every `logging_steps` steps as configured above.
train_result = trainer.train()

# Print a short summary of the training run.
print(train_result)


## 9. Save the Fine-Tuned Model

Saves the LoRA adapter weights (small file size) to `OUTPUT_DIR`. To get a
single standalone model (adapter merged into the base weights), run the
optional merge cell afterward.


In [ ]:
# Saves only the LoRA adapter weights + tokenizer (small, a few MB to ~100MB).
trainer.model.save_pretrained(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)

print(f"Adapter and tokenizer saved to: {OUTPUT_DIR}")


In [ ]:
# ------------------------------------------------------------------------
# OPTIONAL: Merge LoRA adapter into the base model
# ------------------------------------------------------------------------
# Produces a single, standalone fine-tuned model (same size as the base
# model) that can be loaded without the `peft` library. Useful for
# deployment. Requires enough RAM/VRAM to hold the full-precision model.

MERGE_AND_SAVE = False  # COMPULSORY: set True if you want a merged full model

if MERGE_AND_SAVE:
    merged_model = trainer.model.merge_and_unload()
    merged_output_dir = OUTPUT_DIR + "-merged"
    merged_model.save_pretrained(merged_output_dir, safe_serialization=True)
    tokenizer.save_pretrained(merged_output_dir)
    print(f"Merged model saved to: {merged_output_dir}")


## 10. (Optional) Push the Fine-Tuned Model to the Hugging Face Hub

In [ ]:
PUSH_TO_HUB = False  # COMPULSORY: set True to enable pushing to the Hub

# COMPULSORY (only if PUSH_TO_HUB=True): your target Hub repo id,
# e.g. "your-username/your-model-name"
HUB_REPO_ID = "your-username/knatware-finetuned-model"

if PUSH_TO_HUB:
    trainer.model.push_to_hub(HUB_REPO_ID)
    tokenizer.push_to_hub(HUB_REPO_ID)
    print(f"Model pushed to: https://huggingface.co/{HUB_REPO_ID}")


## 11. Quick Inference Test

Loads the fine-tuned adapter on top of the base model and generates a sample
response to sanity-check that fine-tuning worked as expected.


In [ ]:
from transformers import pipeline

# Reload base model + attach the fine-tuned adapter for a clean inference test.
inference_model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config=bnb_config if USE_4BIT else None,
    device_map="auto",
    trust_remote_code=True,
)
inference_model = PeftModel.from_pretrained(inference_model, OUTPUT_DIR)

generator = pipeline(
    "text-generation",
    model=inference_model,
    tokenizer=tokenizer,
)

# COMPULSORY: edit this prompt to test your fine-tuned model on a relevant example.
TEST_PROMPT = "### Instruction:\nExplain what fine-tuning an LLM means in one paragraph.\n\n### Response:\n"

output = generator(
    TEST_PROMPT,
    max_new_tokens=200,     # COMPULSORY: max number of new tokens to generate
    do_sample=True,         # COMPULSORY: enables sampling (vs. greedy decoding)
    temperature=0.7,        # COMPULSORY: randomness of output; lower = more deterministic
    top_p=0.9,               # COMPULSORY: nucleus sampling threshold
)

print(output[0]["generated_text"])


## 12. (Optional) Simple Hyperparameter Tuning Sweep

A minimal example of looping over a small grid of hyperparameters (e.g.
learning rate, LoRA rank) to compare training loss across runs. For serious
tuning at scale, consider `optuna` or W&B Sweeps instead.


In [ ]:
# OPTIONAL: example grid-search sweep over learning rate.
# Each run trains for a very small number of steps just to compare loss —
# extend `num_train_epochs` / remove `max_steps` for a real sweep.

RUN_HYPERPARAMETER_SWEEP = False  # set True to run this cell's sweep

if RUN_HYPERPARAMETER_SWEEP:
    learning_rates_to_try = [1e-4, 2e-4, 3e-4]  # COMPULSORY: values to sweep over
    sweep_results = {}

    for lr in learning_rates_to_try:
        print(f"\n=== Training with learning_rate={lr} ===")
        sweep_args = SFTConfig(
            output_dir=f"{OUTPUT_DIR}-lr{lr}",
            max_steps=20,                # short run, just for comparison
            per_device_train_batch_size=2,
            gradient_accumulation_steps=4,
            learning_rate=lr,
            logging_steps=5,
            bf16=True,
            gradient_checkpointing=True,
            max_length=MAX_SEQ_LENGTH,
            dataset_text_field=DATASET_TEXT_FIELD,
            report_to="none",
        )
        sweep_trainer = SFTTrainer(
            model=model,
            args=sweep_args,
            train_dataset=dataset,
            formatting_func=formatting_func,
            processing_class=tokenizer,
            peft_config=lora_config,
        )
        result = sweep_trainer.train()
        sweep_results[lr] = result.training_loss
        print(f"Final training loss for lr={lr}: {result.training_loss}")

    print("\nSweep summary:", sweep_results)


---

### Notes
- If you hit `CUDA out of memory`, reduce `per_device_train_batch_size`,
  increase `gradient_accumulation_steps`, lower `MAX_SEQ_LENGTH`, or keep
  `USE_4BIT = True`.
- For larger models (7B+), use Colab Pro/Pro+ with an A100 GPU, or reduce
  `LORA_R` and batch size further.
- Always validate the fine-tuned model's outputs before deploying it to
  production.

---
**© Knatware Technology** — Notebook developed by **Kayode Okosi**, LLM Developer.
